# 06 — Write Once, Deploy to LangChain, CrewAI, AutoGen, and Seven More

**The pitch.** Every agent framework has its own prompt format. LangChain wants messages. CrewAI wants role/goal/backstory. AutoGen wants system_message. Migrating a prompt from one to another means rewriting it by hand. mycontext's `Context` object is framework-agnostic: build once, export to ten frameworks with a single method call — or create full agent objects directly.

**Who this is for:** Developers working across multiple agent frameworks, teams evaluating frameworks, anyone who hates rewriting prompts.

**What you'll learn:**
- All 10 export methods: `.to_openai()`, `.to_langchain()`, `.to_crewai()`, `.to_autogen()`, `.to_anthropic()`, `.to_google()`, `.to_llamaindex()`, `.to_prompt()`, `.to_messages()`, `.to_markdown()`
- Integration helpers: `CrewAIHelper.create_agent()`, `AutoGenHelper.create_assistant()`, `LangChainHelper.to_messages()`, `GoogleADKHelper.create_agent()`
- `auto_integrate(ctx, 'crewai')` — dispatch by framework name string
- Quality equivalence: same prompt, same question, same scores across frameworks

**Next:** **07** = Executable skills — the SKILL.md that runs, measures, and improves.

## Research grounding

> **Portability-performance:** The same semantic prompt, adapted to each framework's native format, produces statistically equivalent output quality across OpenAI, Anthropic, CrewAI, and AutoGen — measured via `OutputEvaluator` in mycontext's cross-framework evaluation. Framework choice is therefore a deployment decision, not a quality one.

> **Provider-aware rendering:** `Context.assemble_for_model(provider)` applies provider-specific formatting backed by empirical testing: Anthropic performs better with XML delimiters and positive constraint reframes; Gemini with explicit verbosity anchors and trait adjectives on the role; OpenAI with instruction mirroring for long knowledge blocks *(mycontext provider rendering research, 2026-03-16)*.

## The framework format zoo

| Framework | Format mycontext produces |
|-----------|---------------------------|
| OpenAI | `{messages: [{role, content}], model, temperature}` |
| Anthropic | `{system: str, messages: [...]}` |
| Google Gemini | `{contents: [...], generation_config: {...}}` |
| LangChain | `{system_message: str, human_message: str}` |
| LlamaIndex | `{system_prompt: str, query_instruction: str}` |
| CrewAI | `{role: str, goal: str, backstory: str, expected_output: str}` |
| AutoGen | `{system_message: str, name: str, ...}` |
| DSPy | Signature string |
| Semantic Kernel | Prompt template |
| Google ADK | Instruction string + agent object |

## Install and setup

In [ ]:
# %pip install -q -U "mycontext-ai>=0.10.1"
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dotenv import load_dotenv
from IPython.display import display_markdown

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

def md(s):
    display_markdown(s, raw=True)

import mycontext

md(f"**mycontext-ai** `{getattr(mycontext, '__version__', '?')}`")


## Step 1 — Build one Context (build once)

In [ ]:
from mycontext.intelligence import get_pattern_class

# Use code_reviewer as our example — concrete, well-parameterized
CodeReviewer = get_pattern_class("code_reviewer")

sample_code = """
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)
"""

ctx = CodeReviewer.build_context(
    question="Review this code for security vulnerabilities",
    code=sample_code,
    language="python",
)

md(f"Context built. Template: `{ctx.metadata.get('template', 'code_reviewer')}`")
md(f"Assembled prompt length: {len(ctx.assemble())} chars")


## Step 2 — All 10 export formats (deploy anywhere)

In [ ]:
import json
# OpenAI
openai_fmt = ctx.to_openai()
md("=== to_openai() ===")
_sj = json.dumps(openai_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# Anthropic
anthropic_fmt = ctx.to_anthropic()
md("=== to_anthropic() ===")
_sj = json.dumps(anthropic_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# CrewAI
crewai_fmt = ctx.to_crewai()
md("=== to_crewai() ===")
_sj = json.dumps(crewai_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# AutoGen
autogen_fmt = ctx.to_autogen()
md("=== to_autogen() ===")
_sj = json.dumps(autogen_fmt, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# LangChain, Google Gemini, LlamaIndex
md("=== to_langchain() ===")
_sj = json.dumps(ctx.to_langchain(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

md("\n=== to_google() ===")
_sj = json.dumps(ctx.to_google(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")

md("\n=== to_llamaindex() ===")
_sj = json.dumps(ctx.to_llamaindex(), indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


## Step 3 — Integration helpers: create full agent objects

In [ ]:
import json
from mycontext.integrations.helpers import CrewAIHelper, AutoGenHelper, LangChainHelper

# CrewAI: full agent config
crewai_agent_cfg = CrewAIHelper.create_agent(ctx, name="SecurityReviewer")
md("=== CrewAIHelper.create_agent() ===")
_sj = json.dumps(crewai_agent_cfg, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# AutoGen: assistant config
autogen_agent_cfg = AutoGenHelper.create_assistant(ctx, name="SecurityReviewAgent")
md("=== AutoGenHelper.create_assistant() ===")
_sj = json.dumps(autogen_agent_cfg, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


In [ ]:
import json
# LangChain: prompt messages
lc_msgs = LangChainHelper.to_messages(ctx, user_message="Review the code above")
md("=== LangChainHelper.to_messages() ===")
_sj = json.dumps(lc_msgs, indent=2, default=str)
md("```json\n" + _sj[:12000] + ("..." if len(_sj) > 12000 else "") + "\n```")


## Step 4 — `auto_integrate`: dispatch by name string

In [ ]:
from mycontext.integrations.helpers import auto_integrate

# Same ctx, different framework — by string dispatch
for framework in ["crewai", "autogen", "langchain"]:
    result = auto_integrate(ctx, framework)
    md(f"auto_integrate(ctx, '{framework}'):")
    if isinstance(result, dict):
        md(f"  Keys: {list(result.keys())}")
    else:
        md(f"  Type: {type(result).__name__}")
    md("---")


## Step 5 — Quality equivalence: same score, any framework

In [ ]:
from mycontext.intelligence import QualityMetrics

# The Context quality score is framework-independent
# — it scores the assembled prompt, not the export format
qm = QualityMetrics(mode="heuristic")
score = qm.evaluate(ctx)

md(f"Prompt quality score: {score.overall * 100:.0f}/100")
md("(This score is the same regardless of which framework you export to)")
md("\nOne Context → ten frameworks → identical prompt quality.")
md("Migrate from LangChain to CrewAI without touching the prompt.")


## Summary

| What you learned | API | Key? |
|-----------------|-----|------|
| OpenAI messages format | `ctx.to_openai()` | No |
| Anthropic format | `ctx.to_anthropic()` | No |
| CrewAI agent config | `ctx.to_crewai()` | No |
| AutoGen config | `ctx.to_autogen()` | No |
| LangChain, Google, LlamaIndex | `ctx.to_langchain()`, `.to_google()`, `.to_llamaindex()` | No |
| Full agent creation | `CrewAIHelper.create_agent(ctx)`, `AutoGenHelper.create_assistant(ctx)` | No |
| String dispatch | `auto_integrate(ctx, 'crewai')` | No |

**Next notebook:** [07_executable_skills.ipynb](07_executable_skills.ipynb) — the SKILL.md format that executes, quality-gates, and self-improves.